# SCALAR — cross-modal Sim(3) line-map registration demo

**ScalePluckerNet** (scale-invariant Plücker line matcher) + a **line-based Sim(3) estimator**, run end-to-end on a *real* map pair from our 7-Scenes benchmark: a monocular SLAM line map (unknown scale) registered onto an RGB-D reference map of the `heads` scene.

Runs on **CPU** in under a minute — no dataset download; the map pair ships with the repo.

> Project page: <https://rueyday.github.io/ScalePluckerNet/> · Paper: SCALAR (ICRA 2027 submission)


In [ ]:
# 1) Get the code (and the bundled demo data)
# PLACEHOLDER: replace with the public repo URL if it moves.
REPO_URL = 'https://github.com/rueyday/ScalePluckerNet'
!git clone --depth 1 {REPO_URL} scalar 2>/dev/null || (cd scalar && git pull)
%cd scalar
!pip -q install plotly easydict msgpack 2>/dev/null
import sys, os; sys.path.insert(0, os.getcwd())


In [ ]:
# 2) Load the bundled REAL map pair (7-Scenes `heads`, mono -> RGB-D)
import json, re
import numpy as np
raw = open('docs/demo_real_7scenes.js').read()
D = json.loads(re.search(r'window.REAL_DEMO = (.*);', raw).group(1))
q = np.asarray(D['q'], np.float64)   # (Nq, 6) query segment endpoints [x1 y1 z1 x2 y2 z2]
r = np.asarray(D['r'], np.float64)   # (Nr, 6) reference segment endpoints
q1, q2, r1, r2 = q[:, :3], q[:, 3:], r[:, :3], r[:, 3:]
print(D['label'])
print(f'query {len(q)} lines (scale-ambiguous)   reference {len(r)} lines (metric)')


In [ ]:
# 3) Matcher forward pass (skipped gracefully if the checkpoint is absent)
# PLACEHOLDER: if the checkpoint is not committed to the repo, host it and
# set CKPT_URL to the public download link (e.g. a GitHub release asset).
import torch
from lib.sim3_solver import Sim3Solver, segments_to_plucker
CKPT = 'output/synthetic_v6/synthetic_v6/best_val_checkpoint.pth'
CKPT_URL = ''  # <- paste public checkpoint URL here
if not os.path.exists(CKPT) and CKPT_URL:
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    !wget -q -O {CKPT} {CKPT_URL}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
solver = Sim3Solver((q1, q2), (r1, r2), device=dev)
prob = None
if os.path.exists(CKPT):
    from register import load_network
    p_q = segments_to_plucker(q1 * solver.alpha, q2 * solver.alpha)
    p_r = segments_to_plucker(r1 * solver.alpha, r2 * solver.alpha)
    std = float(p_q[:, :3].std()) + 1e-6
    nrm = lambda x: np.concatenate([x[:, :3] / std, x[:, 3:]], 1).astype(np.float32)
    model = load_network(CKPT, torch.device(dev))
    with torch.no_grad():
        prob, _, _ = model(torch.from_numpy(nrm(p_r)[None]).to(dev),
                           torch.from_numpy(nrm(p_q)[None]).to(dev))
    prob = prob[0]
    print('ScalePluckerNet loaded — full pipeline')
else:
    print('checkpoint not found — running the correspondence-free estimator variant')


In [ ]:
# 4) Solve the Sim(3): scale + rotation + translation
import time
t0 = time.time()
s, R, t, info = solver.register(prob=prob)
print(f'solved in {time.time()-t0:.1f}s on {dev}')
print(f'scale s = {s:.4f}   (ground truth {D["s_gt"]:.4f})')
print('R =', np.round(np.asarray(R), 4).tolist())
print('t =', np.round(np.asarray(t).reshape(3), 4).tolist())


In [ ]:
# 5) Interactive 3D: before vs after
import plotly.graph_objects as go
def seg_trace(S, color, name):
    x, y, z = [], [], []
    for a in S:
        x += [a[0], a[3], None]; y += [a[1], a[4], None]; z += [a[2], a[5], None]
    return go.Scatter3d(x=x, y=y, z=z, mode='lines', line=dict(color=color, width=2), name=name)
Rn, tn, sn = np.asarray(R), np.asarray(t).reshape(3), float(s)
tp = lambda P: (sn * (Rn @ P.T)).T + tn
q_al = np.concatenate([tp(q1), tp(q2)], 1)
fig = go.Figure([seg_trace(r, '#f97316', 'RGB-D reference'),
                 seg_trace(q, '#60a5fa', 'mono query (before)'),
                 seg_trace(q_al, '#4ade80', 'mono query (after SCALAR)')])
fig.update_layout(template='plotly_dark', height=650, scene_aspectmode='data',
                  title='toggle traces in the legend to compare before/after')
fig.show()


## What just happened

1. Both maps were pre-normalized so every solver threshold is scale-free.
2. ScalePluckerNet proposed soft correspondences across the modality/scale gap (or, without the checkpoint, the estimator fell back to correspondence-free direction statistics).
3. The estimator generated rotation candidates (2-pair Procrustes + SO(3) grid, keeping ~14 peaks — never trusting the top-1, because Manhattan flips dominate direction-only scores), filtered candidates by pairwise consistency, solved batched 2-pair (t, s) systems, and **verified every pose against the full maps** with a combined moment + perpendicular residual score.
4. A re-proposal beam and one guarded least-squares refit produced the final estimate.

See the [project page](https://rueyday.github.io/ScalePluckerNet/) for the method details, the full benchmark, and the paper.
